# 00 — Raw Data Inspection

Quick sanity checks on freshly downloaded raw data before running the full pipeline.
Run this before `enso build-dataset` when working with updated data.


In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from src.ingestion import nino_indices, soi, zonal_wind

RAW_DIR = Path('../../data/raw')

In [ ]:
# Load each source independently
nino_df = nino_indices.load(raw_dir=RAW_DIR)
soi_df  = soi.load(raw_dir=RAW_DIR)
zwnd_df = zonal_wind.load(raw_dir=RAW_DIR)

print('Niño indices:', nino_df.shape, '|', nino_df.index[0].date(), '→', nino_df.index[-1].date())
print('SOI:         ', soi_df.shape,  '|', soi_df.index[0].date(),  '→', soi_df.index[-1].date())
print('Zonal wind:  ', zwnd_df.shape, '|', zwnd_df.index[0].date(), '→', zwnd_df.index[-1].date())

In [ ]:
# Check for unexpected missing values in each source
for name, df in [('Niño', nino_df), ('SOI', soi_df), ('ZWND', zwnd_df)]:
    miss = df.isnull().sum()
    if miss.any():
        print(f'{name} — missing values found:')
        print(miss[miss > 0])
    else:
        print(f'{name} — no missing values ✓')

In [ ]:
# Quick plot: Niño 3.4 vs SOI to check anti-correlation (physical sanity)
combined = nino_df[['nino34_anom']].join(soi_df).dropna()
combined = combined['1980':]

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(combined.index, combined['nino34_anom'], color='#d6604d', linewidth=0.9)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('Niño 3.4 (°C)')
axes[0].set_title('Raw data sanity check: Niño 3.4 vs SOI (should anti-correlate)', fontweight='bold')

axes[1].plot(combined.index, combined['soi'], color='#4393c3', linewidth=0.9)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('SOI')

plt.tight_layout()
plt.show()

corr = combined['nino34_anom'].corr(combined['soi'])
print(f'Pearson r(Niño 3.4, SOI) = {corr:.3f}  (expected ≈ −0.7 to −0.9)')